In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
target_table = dbutils.widgets.get("target_table")

from datetime import datetime, timedelta
previous_date = (datetime.strptime(fetch_date, '%Y-%m-%d') - timedelta(days=1)).strftime('%Y-%m-%d')

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW cubhub_ma_src AS
SELECT * FROM (
WITH tmptb_dedup AS (
    SELECT 
        Invoice_Number,
        Episode_ID,
        Account_Balance,
        Payer_Type,
        Program_Name,
        reporting_date,
        Date_Invoice_Became_a_Credit,
        Source_System,
        Reimbursement_Team,
        ROW_NUMBER() OVER (
            PARTITION BY 
                Invoice_Number,
                Episode_ID,
                Account_Balance,
                reporting_date
            ORDER BY Invoice_Number
        ) AS rnb
    FROM {target_table}
    WHERE reporting_date IN ('{fetch_date}','{previous_date}')
),
tmptb AS (
    SELECT * FROM tmptb_dedup WHERE rnb = 1
),
-- CTE 2: Calculate first credit date for each invoice
tmptb_with_credit_date AS (
    SELECT 
        Reimbursement_Team,
        Source_System,
        Episode_ID,
        Invoice_Number,
        Account_Balance,
        Payer_Type,
        Program_Name,
        reporting_date,
        Date_Invoice_Became_a_Credit,
        FIRST_VALUE(Date_Invoice_Became_a_Credit) OVER (
            PARTITION BY Invoice_Number, SIGN(Account_Balance)
            ORDER BY Invoice_Number, reporting_date
        ) AS Date_Invoice_Became_a_Credit_tmp
    FROM tmptb
),
-- CTE 3: Calculate Net Episodic Balance for specific criteria
episode_balances AS (
    SELECT 
        Episode_ID,
        reporting_date,
        Payer_Type,
        SUM(COALESCE(Account_Balance, 0)) AS Net_Episodic_Balance
    FROM {target_table}
    WHERE Episode_ID IS NOT NULL
    AND reporting_date = CAST('{fetch_date}' AS DATE)
    GROUP BY Payer_Type, Episode_ID, reporting_date
),
negative_invoices AS (
    SELECT 
        Episode_ID,
        Invoice_Number,
        Payer_Type,
        Account_Balance,
        reporting_date
    FROM {target_table}
    WHERE Reimbursement_Team = '289'
    AND Payer_Type IN ('HOSPICE - MEDICARE', 'MEDICARE', 'PPS - NON MEDICARE')
    AND Account_Balance < 0
    AND Source_System = 'HCHB'
    AND reporting_date = CAST('{fetch_date}' AS DATE)
),
episode_join AS (
    SELECT 
        eb.Net_Episodic_Balance,
        eb.Payer_Type,
        ni.reporting_date,
        ni.Episode_ID,
        ni.Invoice_Number,
        ni.Account_Balance
    FROM negative_invoices ni
    INNER JOIN episode_balances eb
        ON ni.reporting_date = eb.reporting_date
        AND ni.Episode_ID = eb.Episode_ID
        AND ni.Payer_Type = eb.Payer_Type
),
-- CTE 4: Join with episodic balance data
tmptb1_base AS (
    SELECT 
        m.Reimbursement_Team,
        m.Source_System,
        m.Episode_ID,
        m.Invoice_Number,
        m.Account_Balance,
        m.Payer_Type,
        m.Program_Name,
        m.reporting_date,
        m.Date_Invoice_Became_a_Credit,
        m.Date_Invoice_Became_a_Credit_tmp,
        e.Net_Episodic_Balance
    FROM tmptb_with_credit_date m
    LEFT JOIN episode_join e
        ON m.reporting_date = e.reporting_date
        AND m.Invoice_Number = e.Invoice_Number
        AND m.Account_Balance = e.Account_Balance
),
-- CTE 5: Update credit date for new credits
tmptb1_credit_updated AS (
    SELECT 
        Reimbursement_Team,
        Source_System,
        Episode_ID,
        Invoice_Number,
        Account_Balance,
        Payer_Type,
        Program_Name,
        reporting_date,
        Date_Invoice_Became_a_Credit,
        CASE 
            WHEN Account_Balance < 0 
                AND reporting_date = CAST('{fetch_date}' AS DATE)
                AND Date_Invoice_Became_a_Credit_tmp IS NULL
            THEN CAST(reporting_date AS STRING)
            ELSE Date_Invoice_Became_a_Credit_tmp
        END AS Date_Invoice_Became_a_Credit_tmp,
        Net_Episodic_Balance
    FROM tmptb1_base
),
-- CTE 6: Apply credit date to main field
tmptb1_credit_final AS (
    SELECT 
        Reimbursement_Team,
        Source_System,
        Episode_ID,
        Invoice_Number,
        Account_Balance,
        Payer_Type,
        Program_Name,
        reporting_date,
        CASE 
            WHEN reporting_date = CAST('{fetch_date}' AS DATE)
            AND (Date_Invoice_Became_a_Credit IS NULL)
            AND Account_Balance < 0
            THEN Date_Invoice_Became_a_Credit_tmp
            ELSE Date_Invoice_Became_a_Credit
        END AS Date_Invoice_Became_a_Credit,
        Date_Invoice_Became_a_Credit_tmp,
        Net_Episodic_Balance
    FROM tmptb1_credit_updated
),
-- CTE 7: Clear credit date for positive balances
tmptb1_cleared AS (
    SELECT 
        Reimbursement_Team,
        Source_System,
        Episode_ID,
        Invoice_Number,
        Account_Balance,
        Payer_Type,
        Program_Name,
        reporting_date,
        CASE 
            WHEN Account_Balance > 0 AND reporting_date = CAST('{fetch_date}' AS DATE)
            THEN NULL
            ELSE Date_Invoice_Became_a_Credit
        END AS Date_Invoice_Became_a_Credit,
        Net_Episodic_Balance
    FROM tmptb1_credit_final
),
-- CTE 8: Set Net Episodic Balance for negative invoices without it
tmptb1_net_balance AS (
    SELECT 
        Reimbursement_Team,
        Source_System,
        Episode_ID,
        Invoice_Number,
        Account_Balance,
        Payer_Type,
        Program_Name,
        reporting_date,
        Date_Invoice_Became_a_Credit,
        CASE 
            WHEN Account_Balance < 0 
            AND Net_Episodic_Balance IS NULL 
            AND reporting_date = CAST('{fetch_date}' AS DATE)
            THEN Account_Balance
            ELSE Net_Episodic_Balance
        END AS Net_Episodic_Balance
    FROM tmptb1_cleared
),
-- CTE 9: Zero out positive net episodic balances for specific criteria
tmptb1_zero_positive AS (
    SELECT 
        Reimbursement_Team,
        Source_System,
        Episode_ID,
        Invoice_Number,
        Account_Balance,
        Payer_Type,
        Program_Name,
        reporting_date,
        Date_Invoice_Became_a_Credit,
        CASE 
            WHEN Episode_ID IS NOT NULL
            AND Reimbursement_Team = '289'
            AND Payer_Type IN ('HOSPICE - MEDICARE', 'MEDICARE', 'PPS - NON MEDICARE')
            AND Account_Balance < 0
            AND Source_System = 'HCHB'
            AND Net_Episodic_Balance > 0
            AND reporting_date = CAST('{fetch_date}' AS DATE)
            THEN 0
            ELSE Net_Episodic_Balance
        END AS Net_Episodic_Balance
    FROM tmptb1_net_balance
),
-- CTE 10: Identify duplicate episodes to zero out
episode_duplicates AS (
    SELECT 
        Episode_ID,
        Invoice_Number,
        reporting_date,
        Net_Episodic_Balance,
        ROW_NUMBER() OVER (
            PARTITION BY Episode_ID
            ORDER BY Invoice_Number DESC
        ) AS rnb
    FROM tmptb1_zero_positive
    WHERE Episode_ID IS NOT NULL
    AND Reimbursement_Team = '289'
    AND Payer_Type IN ('HOSPICE - MEDICARE', 'MEDICARE', 'PPS - NON MEDICARE')
    AND Account_Balance < 0
    AND Source_System = 'HCHB'
    AND Net_Episodic_Balance < 0
    AND reporting_date = CAST('{fetch_date}' AS DATE)
),
-- CTE 11: Zero out duplicate episode records
tmptb1_final AS (
    SELECT 
        t.Reimbursement_Team,
        t.Source_System,
        t.Episode_ID,
        t.Invoice_Number,
        t.Account_Balance,
        t.Payer_Type,
        t.Program_Name,
        t.reporting_date,
        t.Date_Invoice_Became_a_Credit,
        CASE 
            WHEN d.Invoice_Number IS NOT NULL 
            AND d.rnb >= 2 
            AND t.reporting_date = d.reporting_date
            THEN 0
            ELSE t.Net_Episodic_Balance
        END AS Net_Episodic_Balance
    FROM tmptb1_zero_positive t
    LEFT JOIN episode_duplicates d
        ON t.Invoice_Number = d.Invoice_Number
        AND t.reporting_date = d.reporting_date
),
-- CTE 12: Calculate final fields
final_calculations AS (
    SELECT 
        Invoice_Number,
        CASE 
            WHEN Payer_Type IN (
                'HOSPICE - MEDICAID', 'HOSPICE - MEDICARE', 'MANAGED - MEDICAID', 
                'MANAGED - MEDICARE', 'MANAGED - MEDICARE - APM', 'MEDICAID', 'MEDICARE',
                'MEDICARE - PART B', 'PPS - NON MEDICARE', 'MCD', 'MMCD', 'MMCR', 
                'SCH', 'CTRG', 'INSG', 'WCOG', 'MVAG'
            )
            OR Program_Name LIKE '%denial%'
            THEN 'Government'
            ELSE 'Non-Government'
        END AS Govt_Non_Govt,
        Date_Invoice_Became_a_Credit,
        CASE 
            WHEN Date_Invoice_Became_a_Credit IS NOT NULL
            THEN DATEDIFF(DAY, TO_DATE(Date_Invoice_Became_a_Credit), reporting_date)
        END AS Age_of_Credit_Balance,
        Net_Episodic_Balance AS Credit_Reporting_Balance
    FROM tmptb1_final
    WHERE reporting_date = CAST('{fetch_date}' AS DATE)
)
SELECT * FROM final_calculations
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {target_table} AS target
USING cubhub_ma_src AS source
ON target.Invoice_Number = source.Invoice_Number
   AND target.reporting_date = CAST('{fetch_date}' AS DATE)

WHEN MATCHED THEN
    UPDATE SET
        target.Govt_Non_Govt = source.Govt_Non_Govt,
        target.Date_Invoice_Became_a_Credit = source.Date_Invoice_Became_a_Credit,
        target.Age_of_Credit_Balance = source.Age_of_Credit_Balance,
        target.Credit_Reporting_Balance = source.Credit_Reporting_Balance;
""")
)

In [0]:
%skip
display(
spark.sql("""
UPDATE {target_table}
SET 
    Age_of_Credit_Balance = ' '
    Credit_Reporting_Balance = ' '
WHERE reporting_date = CAST('{fetch_date}' AS DATE)
AND (Age_of_Credit_Balance IS NULL OR Credit_Reporting_Balance IS NULL);
""")
)